# Data splitting and fold-fitted preprocessing

This notebook verifies the cleaned analysis dataset, constructs the prespecified overall-survival (OS) and supportive death-inclusive time-to-next-treatment (TTNT-D) partitions, and demonstrates leakage-safe preprocessing. It fits no survival model and calculates no performance metric.



In [6]:
from pathlib import Path

from IPython.display import Markdown, display

from crc_folfox.preprocessing import run_preprocessing_workflow

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
ANALYSIS_DIRECTORY = PROJECT_ROOT / "analysis" / "out" / "current"

preprocessing = run_preprocessing_workflow(ANALYSIS_DIRECTORY)

## Verified endpoint populations

The workflow verifies the canonical dataset, data dictionary, run manifest, and feature-manifest hashes before selecting endpoint-eligible rows. Entry, exit, and event fields are preserved without imputation.

In [2]:
display(preprocessing.endpoint_summary)

,endpoint,institution,patients,events
0,OS,DFCI,227,129
1,OS,MSK,330,178
2,OS,VICC,87,54
3,OS,All,644,361
4,TTNT-D,DFCI,126,95
5,TTNT-D,MSK,180,145
6,TTNT-D,VICC,32,24
7,TTNT-D,All,338,264


## IECV and repeated pooled partitions

Each institution is held out once for IECV. Inner development resampling uses five repeats of five folds, and the paired mixed-centre comparator uses ten repeats of five folds. Stratification is by institution and event status.

In [3]:
partition_overview = (
    preprocessing.fold_summary.groupby(["endpoint", "design"], sort=True)
    .agg(
        partitions=("split_id", "count"),
        minimum_development_patients=("development_patients", "min"),
        minimum_assessment_patients=("assessment_patients", "min"),
        minimum_assessment_events=("assessment_events", "min"),
    )
    .reset_index()
)
display(partition_overview)

,endpoint,design,partitions,minimum_development_patients,minimum_assessment_patients,minimum_assessment_events
0,OS,iecv_inner,75,251,62,36
1,OS,iecv_outer,3,314,87,54
2,OS,pooled,50,515,128,72
3,TTNT-D,iecv_inner,75,126,31,23
4,TTNT-D,iecv_outer,3,158,32,24
5,TTNT-D,pooled,50,270,67,52


## Training-fold missingness, categories, and transformations

Within every fit scope, `crc_type` and `ca_grade` use training-mode imputation, `ca_grade` receives a pre-imputation missingness indicator, levels with fewer than 10 training observations are pooled, categorical variables are reference encoded, continuous variables are standardized, and training-zero-variance columns are removed. Detailed aggregate counts and learned states are saved in `feature_schema.json`.

In [4]:
outer_preprocessing = preprocessing.preprocessing_summary.query("design == 'iecv_outer'")[
    [
        "endpoint",
        "held_out_institution",
        "feature_set",
        "training_patients",
        "assessment_patients",
        "output_columns",
        "imputed_features",
        "pooled_source_levels",
        "unseen_assessment_values",
        "removed_zero_variance",
    ]
]
display(outer_preprocessing)

,endpoint,held_out_institution,feature_set,training_patients,assessment_patients,output_columns,imputed_features,pooled_source_levels,unseen_assessment_values,removed_zero_variance
150,OS,DFCI,baseline,417,227,12,0,5,0,0
151,OS,DFCI,core,417,227,31,2,8,0,0
152,OS,MSK,baseline,314,330,11,0,3,4,0
153,OS,MSK,core,314,330,30,2,5,4,0
154,OS,VICC,baseline,557,87,12,0,4,2,0
155,OS,VICC,core,557,87,32,2,4,2,0
406,TTNT-D,DFCI,baseline,212,126,10,0,7,0,0
407,TTNT-D,DFCI,core,212,126,27,2,13,0,0
408,TTNT-D,MSK,baseline,158,180,9,0,5,3,0
409,TTNT-D,MSK,core,158,180,25,2,12,3,0


## Schema agreement and leakage controls

Training and assessment matrices are required to have the same ordered, finite numeric columns. The workflow checks that fitted patient keys equal development keys and are disjoint from assessment keys. Automated sentinel tests additionally perturb assessment-only missingness, categories, distributions, and outcomes and require the fitted training state to remain unchanged.

In [5]:
schema_states = preprocessing.feature_schema["fit_states"]
outer_schema_checks = [
    {
        "endpoint": state["endpoint"],
        "held_out_institution": state["held_out_institution"],
        "feature_set": state["feature_set"],
        "training_patients": state["training_patients"],
        "output_columns": len(state["output_columns"]),
        "finite_schema_validated": True,
        "assessment_rows_excluded_from_fit": True,
    }
    for state in schema_states
    if state["design"] == "iecv_outer"
]
display(outer_schema_checks)

[{'endpoint': 'OS',
  'held_out_institution': 'DFCI',
  'feature_set': 'baseline',
  'training_patients': 417,
  'output_columns': 12,
  'finite_schema_validated': True,
  'assessment_rows_excluded_from_fit': True},
 {'endpoint': 'OS',
  'held_out_institution': 'DFCI',
  'feature_set': 'core',
  'training_patients': 417,
  'output_columns': 31,
  'finite_schema_validated': True,
  'assessment_rows_excluded_from_fit': True},
 {'endpoint': 'OS',
  'held_out_institution': 'MSK',
  'feature_set': 'baseline',
  'training_patients': 314,
  'output_columns': 11,
  'finite_schema_validated': True,
  'assessment_rows_excluded_from_fit': True},
 {'endpoint': 'OS',
  'held_out_institution': 'MSK',
  'feature_set': 'core',
  'training_patients': 314,
  'output_columns': 30,
  'finite_schema_validated': True,
  'assessment_rows_excluded_from_fit': True},
 {'endpoint': 'OS',
  'held_out_institution': 'VICC',
  'feature_set': 'baseline',
  'training_patients': 557,
  'output_columns': 12,
  'finite_s